# 不使用@tool的方式定义工具
## 举例1

In [4]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from rich import print as rprint
# 将env文件中的变量加载为环境变量
#override=True：表示.env优先
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
model = ChatDeepSeek(
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    model_name="deepseek-v4-flash"
)
# 2、声明一个函数（工具）
def get_weather(city:str):
    return f"{city}天气晴朗"
#3、将函数绑定在模型上
model_with_tools=model.bind_tools([get_weather])
#4、调用模型
response=model_with_tools.invoke("北京的天气怎么样")
rprint(response)

AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'The user asks about weather in Beijing. I should call get_weather for Beijing.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 62,
            'prompt_tokens': 348,
            'total_tokens': 410,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 17,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
            'prompt_cache_hit_tokens': 256,
            'prompt_cache_miss_tokens': 92
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
        'id': '92bfa98e-b8a8-4e83-ab38-462099e441dc',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a02d71-e344-70a2-80d5-1b9c821bc1d8-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '北京'},
            'id': 'call_00_7YQUrHM5pkYAlCGP6DmX4721',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 348,
        'output_tokens': 62,
        'total_tokens': 410,
        'input_token_details': {'cache_read': 256},
        'output_token_details': {'reasoning': 17}
    }
)

## 2、工具描述的各部分详解
## 2.1 了解convert_to_openai_tool

执行 model.bind_tools([get_weather]) ，底层最终会调用 convert_to_openai_tool 生成工具描述。所
以我们可以直接调用后者查看解析后的工具描述。

In [7]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city:str):
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.2description说明

In [8]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city:str):
    """
    查询城市的天气
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.3参数的说明

In [9]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city:str):
    """
    查询城市的天气
    args:
        city:具体的城市
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气\nargs:\n    city:具体的城市',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.4参数类型的说明
举例1：正确的

In [10]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city):
    """
    查询城市的天气
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {'properties': {'city': {}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2：错误的
如果在docstring中声明了参数的描述，则必须在函数声明处指明参数的类型

In [11]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city):
    """
    查询城市的天气
    agrs:
        city:具体的城市
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气\nagrs:\n    city:具体的城市',
        'parameters': {'properties': {'city': {}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.5 参数默认值说明
一旦参数设置的默认值，则打印的结果中的required字段中就不再包含此参数。

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city:str="beijing"):
    """
    查询城市的天气
    agrs:
        city:具体的城市
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

In [12]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(dt:str,city:str="beijing"):
    """
    查询城市的天气
    agrs:
        city:具体的城市
        dt:时间
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气\nagrs:\n    city:具体的城市\n    dt:时间',
        'parameters': {
            'properties': {'dt': {'type': 'string'}, 'city': {'default': 'beijing', 'type': 'string'}},
            'required': ['dt'],
            'type': 'object'
        }
    }
}